# 문자열 유사도 기반 Pair 생성

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import itertools

# 상품명 로딩
df = pd.read_csv("product_names.csv")  # 컬럼: product_name
names = df["product_name"].tolist()

# 벡터화 (Char n-gram)
vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 4))
X = vectorizer.fit_transform(names)
sim_matrix = cosine_similarity(X)

# threshold 이상 유사한 상품명을 positive pair로 간주
threshold = 0.8
pairs = []
for i, j in itertools.combinations(range(len(names)), 2):
    if sim_matrix[i, j] >= threshold:
        pairs.append((names[i], names[j]))

# 결과 저장
pd.DataFrame(pairs, columns=["anchor", "positive"]).to_csv("positive_pairs.csv", index=False)


# 임베딩 기반 KNN positive pair 생성 (E5/BGE 활용)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# 모델 & 상품명
model = SentenceTransformer("intfloat/multilingual-e5-base")  # 또는 BGE
df = pd.read_csv("product_names.csv")
names = df["product_name"].tolist()

# 임베딩 추출
embeddings = model.encode(["query: " + n for n in names], convert_to_numpy=True)

# Cosine 유사도 기반 top-3 이웃
sim = cosine_similarity(embeddings)
np.fill_diagonal(sim, 0)

pairs = []
for i, row in enumerate(sim):
    top_idxs = row.argsort()[-3:]  # top-3
    for j in top_idxs:
        pairs.append((names[i], names[j]))

pd.DataFrame(pairs, columns=["anchor", "positive"]).to_csv("positive_pairs_knn.csv", index=False)
